# LLM-Assisted Thematic Analysis of Interview Transcripts

---

This notebook walks you through using a **large language model (LLM)** to perform thematic analysis on interview transcripts. It follows the principles of **Reflexive Thematic Analysis (Braun & Clarke)** — themes emerge from the data rather than being decided in advance.

**No coding experience required.** Every cell is explained before you run it.

---

### Why use an LLM for thematic analysis?

LLMs can help you move through large volumes of transcript data more quickly by:
- Identifying candidate themes and patterns across long interviews
- Surfacing relevant quotes to support each theme
- Producing a structured first-pass analysis you can then review, revise, and build on

The LLM is a **research aid**, not a replacement for your own interpretive judgment. You should always review its output critically.

---

### Why chunking?

Open-source models running locally have a limited **context window** — the amount of text they can read at one time. A typical 60-minute interview transcript (~9,000 words) is too long to analyze in one pass.

This notebook solves that with a **Map-Reduce** pipeline:

| Stage | What happens |
|---|---|
| **1. Chunk** | The transcript is split into overlapping segments of manageable size |
| **2. Map** | The LLM analyzes each segment and notes candidate themes and quotes |
| **3. Reduce** | Those notes are merged and consolidated into a final theme list |

---

### What you need before starting

1. **Transcript files** saved as `.docx` (Microsoft Word) format
2. **Ollama** installed on your computer — [download here](https://ollama.com). This runs the LLM locally so your data never leaves your machine.
3. A model pulled via Ollama. Run this once in your terminal:
   ```
   ollama pull qwen2.5:7b
   ```

> **Privacy note:** This pipeline runs entirely on your own computer. Your transcripts are never sent to the cloud.

---
## Step 1 — Install required packages

Run this cell once. It installs two packages:
- `ollama` — lets Python talk to your locally running LLM
- `python-docx` — lets Python read Word documents

In [ ]:
%pip install ollama python-docx

---
## Step 2 — Configure your settings

Edit the values in this cell before running anything else. This is the **only cell you need to edit**.

- Set `TRANSCRIPTS_FOLDER` to the folder containing your `.docx` transcript files
- Set `OUTPUT_FOLDER` to where you want the analysis results saved
- Set `RESEARCH_FOCUS` to a one or two sentence description of your study — this helps the model understand what kinds of themes are relevant. Leave it blank for a fully open-ended analysis.

**Transcript format note:** Your transcript should use a consistent label for the interviewer (e.g., `Interviewer:`, `Q:`, `Researcher:`). The pipeline will automatically filter out interviewer turns so the model only analyzes the participant's voice.

In [ ]:
# ── EDIT THESE ────────────────────────────────────────────────────────────────

TRANSCRIPTS_FOLDER = "/path/to/your/transcripts"   # folder with your .docx files
OUTPUT_FOLDER      = "/path/to/your/results"        # where results will be saved

RESEARCH_FOCUS = ""  # e.g. "This study explores how early-career teachers experience burnout."
                     # Leave blank for fully open-ended analysis.

# ── MODEL SETTINGS ────────────────────────────────────────────────────────────
# Which Ollama model to use. Must be pulled first via: ollama pull <model-name>
# Recommended: qwen2.5:7b (good balance of speed and quality)
# Alternatives: llama3:8b, mistral, gemma:7b

MODEL       = "qwen2.5:7b"
TEMPERATURE = 0.2    # 0 = very consistent; 1 = more varied. 0.2 works well for analysis.
NUM_CTX     = 16384  # context window size in tokens — leave as-is unless you know your model supports more

# ── CHUNKING SETTINGS ─────────────────────────────────────────────────────────
# These control how the transcript is split into segments.
# The defaults work well for most interviews. Adjust only if needed.

CHUNK_TARGET_WORDS = 2000   # aim for chunks of this many words
CHUNK_MAX_WORDS    = 2800   # never exceed this (prevents context overflow)
MERGE_BATCH_SIZE   = 5      # how many chunk summaries to merge at once

# ──────────────────────────────────────────────────────────────────────────────
print("Settings saved.")

---
## Step 3 — Review the prompts

Prompts are the instructions sent to the LLM at each stage of the pipeline. They define what the model is looking for and how it should respond.

You do not need to edit these to get started. But if you want to tailor the analysis — for example, to focus on a particular aspect of experience — you can adjust the wording here.

There are three prompts, one per pipeline stage:

| Prompt | Stage | What it instructs the model to do |
|---|---|---|
| `CHUNK_PROMPT` | Map | Read one segment and identify candidate themes with supporting quotes |
| `MERGE_PROMPT` | Reduce (intermediate) | Consolidate overlapping themes from multiple segments |
| `SYNTHESIS_PROMPT` | Reduce (final) | Produce the final structured theme list |

In [ ]:
# ── SYSTEM PROMPT ─────────────────────────────────────────────────────────────
# Sent at the start of every LLM call. Sets the model's role and general approach.

_focus_line = f"\n\nResearch focus: {RESEARCH_FOCUS}" if RESEARCH_FOCUS else ""

SYSTEM_PROMPT = f"""You are an experienced qualitative researcher conducting reflexive thematic \
analysis (Braun & Clarke). Your task is to analyze interview transcript data inductively, \
allowing themes to emerge from the participant's own words rather than fitting data to a \
predetermined framework.\
{_focus_line}

Guidelines:
- A theme is a pattern of shared meaning, not just a topic that comes up frequently.
- Ground every theme in direct quotes from the participant, reproduced exactly as spoken.
- Be transparent about your interpretation — briefly explain what each quote reveals.
- Focus only on the participant's voice. Ignore the interviewer."""


# ── CHUNK PROMPT ──────────────────────────────────────────────────────────────
# Sent once per transcript segment. {text} is replaced with the actual segment.

CHUNK_PROMPT = """Read the following interview segment and identify candidate themes.

For each theme you identify:
- Give it a short descriptive name (a phrase, not a single word)
- Write 1-2 sentences explaining what it captures about the participant's experience
- Include at least one direct quote from the participant that illustrates the theme
- Briefly explain how the quote connects to the theme

If a passage contains little analytically relevant content, note that briefly and move on.
Your notes will be combined with notes from other segments in the next stage.

Interview segment:
---
{text}
---"""


# ── MERGE PROMPT ──────────────────────────────────────────────────────────────
# Used to consolidate theme notes from multiple segments into one.
# {count} and {notes} are filled in automatically.

MERGE_PROMPT = """Below are theme notes from {count} segments of the same interview.

Consolidate these into a single set of themes by:
- Merging themes that capture the same underlying idea (keep the clearest name)
- Splitting any themes that bundle together distinct ideas
- Keeping the best supporting quotes for each theme
- Noting when a theme appears across multiple segments (this suggests it is central)
- Preserving any tensions or contradictions in the data — do not flatten them

Segment notes:
---
{notes}
---

Output a single consolidated list of themes using the same structure:
theme name → description → supporting quote(s) → interpretation."""


# ── SYNTHESIS PROMPT ──────────────────────────────────────────────────────────
# The final step. Turns consolidated theme notes into a clean, structured report.
# {notes} is filled in automatically.

SYNTHESIS_PROMPT = """You have theme notes covering an entire interview. Produce a final \
thematic analysis report structured as follows.

For each theme, use this format:

  Theme [N]: [Descriptive Theme Name]

  [2-3 sentences describing what this theme captures about the participant's experience
  and why it is analytically significant.]

  Supporting evidence:
  > [Exact quote from the participant]
  [1-2 sentences interpreting this quote in relation to the theme.]

  > [Second quote, if available]
  [Interpretation.]

Rules:
- List all themes, ordered from most to least central to the participant's account
- Quotes must be reproduced exactly as spoken — do not paraphrase
- Theme names must be descriptive phrases, not single words
- Do not add a narrative summary or conclusion section

Theme notes:
---
{notes}
---"""


print("Prompts ready.")

---
## Step 4 — Load the pipeline functions

Run this cell as-is. It loads all the functions that power the pipeline. You do not need to edit anything here.

Here is what each function does in plain language:

| Function | What it does |
|---|---|
| `read_docx` | Opens a Word file and extracts the text paragraph by paragraph |
| `clean_transcript` | Removes interviewer turns, filler words, timestamps, and blank lines |
| `chunk_transcript` | Splits the cleaned text into segments that fit within the model's context window |
| `call_llm` | Sends a prompt to the model and returns its response |
| `analyze_chunk` | Runs the Map stage on one segment |
| `merge_notes` | Merges a batch of theme notes into one (intermediate Reduce step) |
| `reduce_notes` | Repeats merging until all notes fit in one final call |
| `synthesize_themes` | Runs the final Reduce stage and produces the theme list |
| `save_output` | Writes the results to a text file |

In [ ]:
import ollama
import os
import re
import json
import time
from datetime import datetime
from docx import Document


# ── Patterns used to clean the transcript ─────────────────────────────────────

# Identifies lines that are interviewer turns (adjust the labels to match your transcripts)
_INTERVIEWER = re.compile(
    r'^(interviewer|researcher|moderator|facilitator|Q|I)\s*[:>\-]',
    re.IGNORECASE
)
# Identifies short filler responses with no analytical content
_FILLER = re.compile(
    r'^(mm+[-\s]?hm+|uh[-\s]?hu+h?|yeah|yep|yup|okay|ok|right|sure|got it|I see|go ahead)\s*[.,!?]*$',
    re.IGNORECASE
)
# Identifies timestamp lines (e.g. [0:35] or 1:22:04)
_TIMESTAMP = re.compile(r'^\[?\d{1,2}:\d{2}(:\d{2})?\]?\s*$')


# ── Reading & cleaning ────────────────────────────────────────────────────────

def read_docx(filepath):
    """Extract non-empty paragraphs from a Word document."""
    doc = Document(filepath)
    return [p.text.strip() for p in doc.paragraphs if p.text.strip()]


def clean_transcript(paragraphs):
    """
    Remove content that adds noise without analytical value:
      - Interviewer turns
      - Filler responses (yeah, mm-hmm, okay, etc.)
      - Timestamps
      - Lines shorter than 4 words
    """
    cleaned = []
    for p in paragraphs:
        if _INTERVIEWER.match(p): continue
        if _FILLER.match(p):      continue
        if _TIMESTAMP.match(p):   continue
        if len(p.split()) < 4:    continue
        cleaned.append(p)
    return cleaned


# ── Chunking ──────────────────────────────────────────────────────────────────

def chunk_transcript(paragraphs):
    """
    Group paragraphs into chunks of roughly CHUNK_TARGET_WORDS words.
    Whole paragraphs are kept together so quotes are never cut mid-sentence.
    Very long paragraphs are split at sentence boundaries if needed.
    """
    _sentence_split = re.compile(r'(?<=[.!?])\s+(?=[A-Z])')
    chunks, current, count = [], [], 0

    for para in paragraphs:
        n = len(para.split())

        # If one paragraph is longer than the max, split it at sentence boundaries
        if n > CHUNK_MAX_WORDS:
            if current:
                chunks.append("\n\n".join(current))
                current, count = [], 0
            sub, sub_n = [], 0
            for sentence in _sentence_split.split(para):
                s_n = len(sentence.split())
                if sub_n + s_n > CHUNK_MAX_WORDS and sub:
                    chunks.append(" ".join(sub))
                    sub, sub_n = [], 0
                sub.append(sentence)
                sub_n += s_n
            if sub:
                chunks.append(" ".join(sub))
            continue

        # Flush the current chunk if adding this paragraph would exceed the target
        if count + n > CHUNK_TARGET_WORDS and current:
            chunks.append("\n\n".join(current))
            current, count = [], 0

        current.append(para)
        count += n

    if current:
        chunks.append("\n\n".join(current))

    return chunks


# ── LLM communication ─────────────────────────────────────────────────────────

def call_llm(user_message, retries=3):
    """
    Send a message to the LLM and return its text response.
    Retries up to 3 times if the call fails.

    Every call includes the SYSTEM_PROMPT, which sets the model's role.
    The user_message is the specific instruction for this call.
    """
    for attempt in range(1, retries + 1):
        try:
            response = ollama.chat(
                model=MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": user_message}
                ],
                options={"temperature": TEMPERATURE, "num_ctx": NUM_CTX}
            )
            content = response["message"]["content"]
            if not content or not content.strip():
                raise ValueError("Model returned an empty response.")
            return content
        except Exception as e:
            if attempt < retries:
                print(f"    Attempt {attempt} failed ({e}). Retrying...")
                time.sleep(2)
            else:
                raise RuntimeError(f"LLM call failed after {retries} attempts: {e}") from e


# ── Pipeline stages ───────────────────────────────────────────────────────────

def analyze_chunk(chunk, chunk_num, total):
    """Map stage: identify candidate themes in one transcript segment."""
    print(f"    Analyzing chunk {chunk_num}/{total}...")
    return call_llm(CHUNK_PROMPT.format(text=chunk))


def merge_notes(batch):
    """Merge a batch of theme note sets into one consolidated set."""
    formatted = "\n\n".join(f"[Segment {i+1}]\n{notes}" for i, notes in enumerate(batch))
    return call_llm(MERGE_PROMPT.format(count=len(batch), notes=formatted))


def reduce_notes(all_notes):
    """
    Progressively merge theme notes until they fit in a single synthesis call.
    This handles transcripts that generate more chunk summaries than the model
    can process at once.
    """
    current = list(all_notes)
    round_n = 0
    while len(current) > MERGE_BATCH_SIZE:
        round_n += 1
        print(f"  Merge round {round_n}: consolidating {len(current)} sets of notes...")
        next_level = []
        for i in range(0, len(current), MERGE_BATCH_SIZE):
            batch = current[i:i + MERGE_BATCH_SIZE]
            next_level.append(merge_notes(batch) if len(batch) > 1 else batch[0])
        current = next_level
    return current


def synthesize_themes(notes):
    """Final Reduce stage: produce the structured theme list from consolidated notes."""
    print(f"  Writing final theme list...")
    formatted = "\n\n".join(
        f"[Section {i+1} of {len(notes)}]\n{n}" for i, n in enumerate(notes)
    )
    return call_llm(SYNTHESIS_PROMPT.format(notes=formatted))


# ── Output ────────────────────────────────────────────────────────────────────

def save_output(filename, chunk_notes, theme_list):
    """
    Save results to a .txt file in OUTPUT_FOLDER.
    The final theme list appears first.
    Chunk-level notes follow for auditing — so you can trace every theme
    back to the specific transcript segment it came from.
    """
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)
    base     = os.path.splitext(filename)[0]
    out_path = os.path.join(OUTPUT_FOLDER, f"{base}_themes.txt")

    with open(out_path, "w", encoding="utf-8") as f:
        f.write(f"THEMATIC ANALYSIS\n")
        f.write(f"File:   {filename}\n")
        f.write(f"Date:   {datetime.now().strftime('%Y-%m-%d %H:%M')}\n")
        f.write(f"Model:  {MODEL}  |  Temp: {TEMPERATURE}\n")
        f.write(f"Chunks: {len(chunk_notes)}  |  Chunk size: ~{CHUNK_TARGET_WORDS} words\n")
        if RESEARCH_FOCUS:
            f.write(f"Focus:  {RESEARCH_FOCUS}\n")
        f.write("=" * 60 + "\n\n")

        f.write("THEMES\n")
        f.write("-" * 60 + "\n\n")
        f.write(theme_list)

        f.write("\n\n" + "=" * 60 + "\n")
        f.write("CHUNK-LEVEL NOTES (for auditing)\n")
        f.write("-" * 60 + "\n\n")
        for i, notes in enumerate(chunk_notes, 1):
            f.write(f"--- Chunk {i} of {len(chunk_notes)} ---\n{notes}\n\n")

    return out_path


def already_done(filename):
    """Return True if this transcript has already been analyzed."""
    base = os.path.splitext(filename)[0]
    return os.path.exists(os.path.join(OUTPUT_FOLDER, f"{base}_themes.txt"))


print("Pipeline functions loaded.")

---
## Step 5 — Run the analysis

Run the cell below. It will process every `.docx` file in your transcripts folder.

**What to expect:**
- Each transcript is processed one at a time
- Progress is printed as each chunk is analyzed
- Transcripts that have already been analyzed are skipped automatically
- Results are saved as `<filename>_themes.txt` in your output folder

**Estimated time:** A 60-minute transcript takes roughly 5–15 minutes depending on your machine and model. Larger models take longer but produce richer analysis.

> **Tip:** If a run is interrupted, just re-run this cell. Already-completed transcripts will be skipped. To re-analyze a transcript, delete its `_themes.txt` file from the output folder.

In [ ]:
def run_pipeline(filepath, filename):
    """Run the full thematic analysis pipeline on one transcript file."""

    # 1. Read and clean
    raw     = read_docx(filepath)
    cleaned = clean_transcript(raw)

    raw_words     = sum(len(p.split()) for p in raw)
    cleaned_words = sum(len(p.split()) for p in cleaned)
    removed_pct   = (raw_words - cleaned_words) / raw_words * 100 if raw_words else 0

    print(f"  Raw transcript:   {raw_words:,} words")
    print(f"  After cleaning:   {cleaned_words:,} words  ({removed_pct:.0f}% removed as interviewer/filler)")

    # 2. Chunk
    if cleaned_words <= CHUNK_MAX_WORDS:
        chunks = ["\n\n".join(cleaned)]
        print(f"  Short transcript — analyzing as a single chunk\n")
    else:
        chunks = chunk_transcript(cleaned)
        print(f"  Split into {len(chunks)} chunks\n")

    # 3. Map — analyze each chunk
    chunk_notes = [analyze_chunk(chunk, i+1, len(chunks)) for i, chunk in enumerate(chunks)]
    print()

    # 4. Reduce — merge notes if needed, then synthesize
    reduced = chunk_notes if len(chunk_notes) <= MERGE_BATCH_SIZE else reduce_notes(chunk_notes)
    theme_list = synthesize_themes(reduced)

    return chunk_notes, theme_list


# ── Main ──────────────────────────────────────────────────────────────────────

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

all_files = sorted(f for f in os.listdir(TRANSCRIPTS_FOLDER) if f.endswith(".docx"))
pending   = [f for f in all_files if not already_done(f)]
skipped   = [f for f in all_files if already_done(f)]

print(f"{'='*55}")
print(f"  Transcripts found:    {len(all_files)}")
print(f"  Already analyzed:     {len(skipped)}  (skipping)")
print(f"  To be analyzed:       {len(pending)}")
print(f"  Model: {MODEL}")
print(f"{'='*55}\n")

if not pending:
    print("All transcripts have already been analyzed.")
    print("Delete a transcript's _themes.txt file to re-run it.")
else:
    log = []
    for i, filename in enumerate(pending, 1):
        print(f"[{i}/{len(pending)}] {filename}")
        try:
            filepath = os.path.join(TRANSCRIPTS_FOLDER, filename)
            chunk_notes, theme_list = run_pipeline(filepath, filename)
            out_path = save_output(filename, chunk_notes, theme_list)
            print(f"  Saved: {out_path}\n")
            log.append({"file": filename, "status": "success", "output": out_path})
        except Exception as e:
            print(f"  Error processing {filename}: {e}\n")
            log.append({"file": filename, "status": "error", "error": str(e)})

    # Save a log of this run
    log_path = os.path.join(OUTPUT_FOLDER, "run_log.json")
    existing = json.load(open(log_path)) if os.path.exists(log_path) else []
    with open(log_path, "w") as f:
        json.dump(existing + log, f, indent=2)

    success = sum(1 for r in log if r["status"] == "success")
    print(f"{'='*55}")
    print(f"  Done: {success}/{len(pending)} transcripts analyzed.")
    print(f"  Results saved to: {OUTPUT_FOLDER}")
    print(f"{'='*55}")

---
## Frequently asked questions

**Q: My interviewer label is different (e.g., `PI:` or `Moderator:`). Will it be filtered out?**  
The filter recognizes `Interviewer`, `Researcher`, `Moderator`, `Facilitator`, `Q`, and `I` followed by a colon or dash. If your label is something else, open the functions cell above and add it to the `_INTERVIEWER` pattern.

---

**Q: How do I know the themes are accurate?**  
Every output file includes the chunk-by-chunk notes at the bottom, so you can trace any theme back to the exact transcript passage it came from. The model is also instructed to always quote the participant verbatim — if a theme has no supporting quote, treat it with skepticism.

---

**Q: Can I analyze focus groups or group interviews?**  
Yes, but the cleaning step will only filter a single interviewer label. If your transcript has multiple speakers, you may want to either (a) keep all voices and let the model analyze the group as a whole, or (b) split the transcript by speaker before running the pipeline.

---

**Q: Which model should I use?**  
For most research purposes, `qwen2.5:7b` offers a good balance of speed and analytical depth on a modern laptop. If you have a more powerful machine, `llama3:8b` or `mistral` can produce richer output. Avoid the smallest models (e.g., `3b`) for analytical tasks — they tend to produce shallower themes.

---

**Q: What if the model keeps running out of memory or crashing?**  
Reduce `CHUNK_TARGET_WORDS` and `CHUNK_MAX_WORDS` in Step 2, and reduce `NUM_CTX` to `8192`. Smaller chunks and a smaller context window use less memory.

---

**Q: Can I use this with a cloud model (e.g., GPT-4, Claude)?**  
The pipeline can be adapted for cloud APIs, but that requires changing the `call_llm` function to use a different API client. The chunking and merging logic stays the same — those are needed regardless of which model you use, since transcripts are long.